# TCC-04 — Sprint 2 (continuação): v6, v5 com fotos clínicas sobreamostradas + limiar por modalidade

**Projeto:** Tonare — classificador de melanoma (mel vs nv) com fairness por tom de pele
**Framework:** PyTorch | **Treino real:** Kaggle Notebooks (GPU) | **Local (CPU):** só smoke test

## O que muda nesta versão (só duas coisas em relação à v5)

1. **Fotos clínicas do MILK10k repetidas 4x no treino** (`scripts/build_v6_manifest.py`): ~4% do treino
   na v5 -> ~16% na v6. Hipótese: a v4 (clínicas ~10%) foi a melhor em foto clínica (AUC 0,79) e a v5
   (BCN20000 diluiu para ~4%) foi a melhor em dermatoscopia real (MSK 2020, AUC 0,76); sobreamostrar
   as clínicas pode manter os dois ganhos. **Não está provado** — é isso que a v6 testa.
2. **Limiares por modalidade**, escolhidos só na validação (célula 8) e salvos no checkpoint em
   `thresholds_by_modality`. O limiar único em `threshold` continua salvo, como antes.

Splits de val/teste/`test_milk` idênticos aos da v4/v5 (a comparação v4 x v5 x v6 é direta).
O augmentation continua aleatório a cada época, então as cópias repetidas não são vistas iguais.

### Como rodar no Kaggle
Basta ADICIONAR `split_index_v6.csv` ao dataset `tonare-tcc1` (as imagens são as mesmas da v5).
Ligue GPU + Internet e use Run All.


In [ ]:
import os
import sys
from pathlib import Path

# Local: importa do repositório. No Kaggle: cole aqui o conteúdo de scripts/torch_dataset.py
if not os.path.exists("/kaggle/input"):
    sys.path.append(str(Path("..") / "scripts"))
    from torch_dataset import CLASS_NAMES, LABEL_MAP, build_dataloaders, build_datasets


In [ ]:
IS_KAGGLE = os.path.exists("/kaggle/input")

SMOKE_TEST = not IS_KAGGLE      # local (CPU) = smoke test; Kaggle (GPU) = treino real
ARCHITECTURE = "resnet50"       # ou "efficientnet_b0"
SEED = 42

EPOCHS_STAGE1 = 1 if SMOKE_TEST else 5    # máximo; early stopping pode parar antes
EPOCHS_STAGE2 = 1 if SMOKE_TEST else 12   # máximo; early stopping pode parar antes
PATIENCE = 3                              # épocas sem melhora de AUC antes de parar
BATCH_SIZE = 4 if SMOKE_TEST else 32
LIMIT_PER_SPLIT = 40 if SMOKE_TEST else None
NUM_WORKERS = 0 if SMOKE_TEST else 4

MEL_EXTRA_MULTIPLIER = 1.0   # peso extra pra mel ALÉM do balanceamento (mexer só depois de ver resultados)
TARGET_RECALL_MEL = 0.90     # meta clínica de sensibilidade, usada só pra escolher o limiar na VALIDAÇÃO

print(f"IS_KAGGLE={IS_KAGGLE} | SMOKE_TEST={SMOKE_TEST} | arquitetura={ARCHITECTURE} | batch_size={BATCH_SIZE}")


In [ ]:
import copy
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.utils.class_weight import compute_class_weight
from torch import optim
from torchvision import models

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")


## 1. Dados e dataloader

No Kaggle o dataset chega sem a pasta `data/` por cima (`.../processed/...`), então criamos um
link simbólico em `/kaggle/working/data/processed` para o código achar `data/processed/...`.

In [ ]:
if IS_KAGGLE:
    KAGGLE_PROCESSED = "/kaggle/input/datasets/sofyaoliveiras/tonare-tcc1/processed"
    os.makedirs("/kaggle/working/data", exist_ok=True)
    link = "/kaggle/working/data/processed"
    if os.path.islink(link) or os.path.exists(link):
        os.remove(link)
    os.symlink(KAGGLE_PROCESSED, link)
    BASE_DIR = Path("/kaggle/working")
    OUT_DIR = Path("/kaggle/working")
else:
    BASE_DIR = Path("..")
    OUT_DIR = BASE_DIR / "data" / "processed"

SPLIT_INDEX_PATH = BASE_DIR / "data" / "processed" / "split_index_v6.csv"
assert SPLIT_INDEX_PATH.exists(), (
    f"{SPLIT_INDEX_PATH} não encontrado - rode scripts/build_v6_manifest.py primeiro "
    "(local) ou suba a versão nova do dataset no Kaggle (remoto)."
)
print("BASE_DIR:", BASE_DIR, "| OUT_DIR:", OUT_DIR, "| manifesto:", SPLIT_INDEX_PATH.name)


In [ ]:
datasets = build_datasets(base_dir=BASE_DIR, split_index_path=SPLIT_INDEX_PATH,
                          limit_per_split=LIMIT_PER_SPLIT, use_precomputed_aug=False, strong_aug=True)
loaders = build_dataloaders(datasets, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

for split, ds in datasets.items():
    contagem = ds.df["dx"].value_counts().to_dict()
    print(f"{split}: {len(ds)} imagens {contagem}")


In [ ]:
imgs, labels = next(iter(loaders["train"]))
print("Shape do batch de imagens:", imgs.shape)
print("Labels do batch:", labels.tolist())
print("Range de valores:", float(imgs.min()), "a", float(imgs.max()))

assert imgs.shape[1:] == (3, 224, 224), "dataloader não está entregando imagens 224x224x3"
print("\nDataloader validado: uma iteração de teste passou sem erro.")


## 2. Modelo pré-treinado

In [ ]:
def build_model(architecture: str, num_classes: int = 2, pretrained: bool = True) -> nn.Module:
    if architecture == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        model = models.resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif architecture == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    else:
        raise ValueError(f"Arquitetura não suportada: {architecture}")
    return model


def set_backbone_trainable(model: nn.Module, architecture: str, trainable: bool) -> None:
    # Congela/descongela tudo, MENOS a camada final de classificação.
    for param in model.parameters():
        param.requires_grad = trainable
    head = model.fc if architecture == "resnet50" else model.classifier
    for param in head.parameters():
        param.requires_grad = True


model = build_model(ARCHITECTURE).to(DEVICE)
set_backbone_trainable(model, ARCHITECTURE, trainable=False)

n_treinaveis = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total = sum(p.numel() for p in model.parameters())
print(f"Parâmetros treináveis (Estágio 1): {n_treinaveis:,} de {n_total:,} total")


## 3. Loss ponderada

O treino agora tem só imagens originais (~1:6 mel:nv), então o peso "balanced" já dá ~6x mais
importância ao erro em mel. `MEL_EXTRA_MULTIPLIER` (default 1.0) permite reforçar ainda mais.

In [ ]:
train_labels = datasets["train"].df["dx"].map(LABEL_MAP).values
base_weights = compute_class_weight(class_weight="balanced", classes=np.array([0, 1]), y=train_labels)
class_weights = base_weights.copy()
class_weights[1] *= MEL_EXTRA_MULTIPLIER
criterion = nn.CrossEntropyLoss(weight=torch.tensor(class_weights, dtype=torch.float32).to(DEVICE))

print(f"Distribuição do treino: {pd.Series(train_labels).value_counts().to_dict()} (0=nv, 1=mel)")
print(f"Pesos da loss: nv={class_weights[0]:.3f}, mel={class_weights[1]:.3f}")
if not SMOKE_TEST:  # o smoke test é balanceado por construção (20 mel/20 nv)
    assert class_weights[1] > class_weights[0], "peso de mel deveria ser maior que o de nv"


## 4. Loop de treino (early stopping pela AUC de validação)

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None) -> dict:
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss, all_probs, all_labels = 0.0, [], []
    with torch.set_grad_enabled(is_train):
        for imgs, labels in loader:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            outputs = model(imgs)
            loss = criterion(outputs, labels)

            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * imgs.size(0)
            all_probs.extend(F.softmax(outputs.detach(), dim=1)[:, 1].cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    labels_arr, probs_arr = np.array(all_labels), np.array(all_probs)
    preds = (probs_arr >= 0.5).astype(int)
    report = classification_report(labels_arr, preds, target_names=CLASS_NAMES,
                                    output_dict=True, zero_division=0)
    auc = roc_auc_score(labels_arr, probs_arr) if len(set(all_labels)) > 1 else float("nan")
    return {"loss": total_loss / len(loader.dataset), "accuracy": report["accuracy"],
            "recall_mel": report["mel"]["recall"], "auc": auc,
            "labels": labels_arr, "probs": probs_arr}


def train_stage(model, loaders, criterion, optimizer, n_epochs: int, stage_name: str, patience: int = PATIENCE):
    history, best_auc, best_state, sem_melhora = [], -1.0, None, 0
    print(f"[{stage_name}] pesos da loss (nv, mel): {[round(w, 3) for w in criterion.weight.tolist()]}")
    for epoch in range(1, n_epochs + 1):
        tr = run_epoch(model, loaders["train"], criterion, optimizer)
        va = run_epoch(model, loaders["val"], criterion, optimizer=None)
        print(f"[{stage_name}] época {epoch}/{n_epochs} | "
              f"treino: loss={tr['loss']:.3f} auc={tr['auc']:.3f} | "
              f"val: loss={va['loss']:.3f} auc={va['auc']:.3f} recall_mel@0.5={va['recall_mel']:.3f}")
        history.append({"epoch": epoch, "train": tr, "val": va})

        if va["auc"] > best_auc:
            best_auc, best_state, sem_melhora = va["auc"], copy.deepcopy(model.state_dict()), 0
        else:
            sem_melhora += 1
            if sem_melhora >= patience:
                print(f"[{stage_name}] early stopping: sem melhora da AUC de validação há {patience} épocas")
                break

    if best_state is not None:
        model.load_state_dict(best_state)
        print(f"[{stage_name}] modelo restaurado para a melhor época (val AUC = {best_auc:.3f})")
    return history


## 5. Estágio 1 — só a camada final
Backbone congelado, LR alto (1e-3) porque são poucos parâmetros novos.

In [ ]:
optimizer_stage1 = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3, weight_decay=1e-2)
history_stage1 = train_stage(model, loaders, criterion, optimizer_stage1, EPOCHS_STAGE1, "Estágio 1")


## 6. Estágio 2 — fine-tuning profundo
Backbone descongelado, LR bem baixo (1e-5) + weight decay para regularizar.

In [ ]:
set_backbone_trainable(model, ARCHITECTURE, trainable=True)
n_treinaveis = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Parâmetros treináveis (Estágio 2): {n_treinaveis:,}")

optimizer_stage2 = optim.AdamW(model.parameters(), lr=1e-5, weight_decay=1e-2)
history_stage2 = train_stage(model, loaders, criterion, optimizer_stage2, EPOCHS_STAGE2, "Estágio 2")


## 7. Escolha do limiar (só na validação)

O modelo dá uma probabilidade de melanoma; o limiar decide a partir de quanto marcamos como `mel`.
Baixar o limiar aumenta a sensibilidade (menos melanomas perdidos) e custa especificidade
(mais nevos benignos marcados). A tabela mostra esse custo.

In [ ]:
def sens_spec_prec(labels, probs, threshold):
    preds = (probs >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(labels, preds, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else float("nan")
    spec = tn / (tn + fp) if (tn + fp) else float("nan")
    prec = tp / (tp + fp) if (tp + fp) else float("nan")
    return sens, spec, prec


val_res = run_epoch(model, loaders["val"], criterion)
val_labels, val_probs = val_res["labels"], val_res["probs"]
print(f"Validação: AUC-ROC = {val_res['auc']:.3f}\n")

print(f"{'limiar':>7} {'sensibilidade(mel)':>19} {'especificidade':>15} {'precisão(mel)':>14}")
for t in np.arange(0.05, 0.96, 0.05):
    sens, spec, prec = sens_spec_prec(val_labels, val_probs, t)
    print(f"{t:>7.2f} {sens:>19.3f} {spec:>15.3f} {prec:>14.3f}")


## 8. Resultado final no teste (olhado uma única vez)

Limiar escolhido na **validação**: o maior limiar que ainda atinge `TARGET_RECALL_MEL`
(maior limiar = menos falsos positivos dentro da meta). Se nenhum atinge, usa o de maior
sensibilidade e avisa.

In [ ]:
thresholds = np.round(np.arange(0.01, 0.99, 0.01), 2)
sens_por_t = {t: sens_spec_prec(val_labels, val_probs, t)[0] for t in thresholds}
candidatos = [t for t, s in sens_por_t.items() if s >= TARGET_RECALL_MEL]

if candidatos:
    best_threshold = max(candidatos)
    print(f"Limiar escolhido na validação: {best_threshold:.2f} (sensibilidade >= {TARGET_RECALL_MEL})")
else:
    best_threshold = max(sens_por_t, key=lambda t: (sens_por_t[t], t))  # maior sensibilidade; empate -> maior limiar
    print(f"AVISO: nenhum limiar atinge sensibilidade {TARGET_RECALL_MEL} na validação. "
          f"Usando o de maior sensibilidade: {best_threshold:.2f} ({sens_por_t[best_threshold]:.3f})")

test_res = run_epoch(model, loaders["test"], criterion)
test_labels, test_probs = test_res["labels"], test_res["probs"]
print(f"\nTESTE — AUC-ROC = {test_res['auc']:.3f}")
for nome, t in [("limiar padrão 0.50", 0.5), (f"limiar escolhido {best_threshold:.2f}", best_threshold)]:
    sens, spec, prec = sens_spec_prec(test_labels, test_probs, t)
    tn, fp, fn, tp = confusion_matrix(test_labels, (test_probs >= t).astype(int), labels=[0, 1]).ravel()
    print(f"  {nome}: sensibilidade(mel)={sens:.3f} especificidade={spec:.3f} precisão(mel)={prec:.3f} "
          f"| melanomas perdidos: {fn} de {tp + fn} | nevos marcados como mel: {fp} de {tn + fp}")


# --- Limiares POR MODALIDADE (escolhidos só na validação; o teste não entra aqui) ---
# loaders["val"] não embaralha, então a ordem das probabilidades é a ordem de datasets["val"].df
val_mod = datasets["val"].df["image_type"].values
assert len(val_mod) == len(val_labels), "ordem da validação não bate com o dataframe"

thr_mod = {}
for mod in ("dermoscopic", "clinical"):
    m = val_mod == mod
    lab, prb = val_labels[m], val_probs[m]
    sens_t = {t: sens_spec_prec(lab, prb, t)[0] for t in thresholds}
    ok = [t for t, s in sens_t.items() if s >= TARGET_RECALL_MEL]
    thr_a = max(ok) if ok else max(sens_t, key=lambda t: (sens_t[t], t))      # sensibilidade >= meta
    youden = {t: sum(sens_spec_prec(lab, prb, t)[:2]) - 1 for t in thresholds}
    thr_b = max(youden, key=lambda t: (youden[t], t))                          # equilibrado (Youden)
    thr_mod[mod] = {"A": float(thr_a), "B": float(thr_b)}
    print(f"Validação {mod}: {int(lab.sum())} mel / {int((lab == 0).sum())} nv -> "
          f"limiar A (sens>={TARGET_RECALL_MEL})={thr_a:.2f} | limiar B (Youden)={thr_b:.2f}")

print()
print("AVISO: a validação clínica tem poucos casos; trate o limiar clínico como ruidoso.")
print("Teste do HAM10000 (só dermatoscopia) com os limiares de dermatoscopia:")
for nome in ("A", "B"):
    t = thr_mod["dermoscopic"][nome]
    sens, spec, prec = sens_spec_prec(test_labels, test_probs, t)
    print(f"  {nome} (limiar {t:.2f}): sensibilidade={sens:.3f} especificidade={spec:.3f} precisão={prec:.3f}")


In [ ]:
checkpoint_path = OUT_DIR / f"model_{ARCHITECTURE}_baseline_v6.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "architecture": ARCHITECTURE,
    "label_map": LABEL_MAP,
    "threshold": float(best_threshold),
    "target_recall_mel": TARGET_RECALL_MEL,
    "mel_extra_multiplier": MEL_EXTRA_MULTIPLIER,
    "thresholds_by_modality": thr_mod,
    "val_auc": float(val_res["auc"]),
    "smoke_test": SMOKE_TEST,
}, checkpoint_path)
print(f"Checkpoint salvo em: {checkpoint_path}")
if SMOKE_TEST:
    print("SMOKE TEST (CPU, amostra pequena): serve só para provar que o pipeline roda. "
          "Rode no Kaggle (GPU) para o treino de verdade.")


## Próximo passo
Baixar `model_resnet50_baseline_v6.pt` para `data/processed/` e comparar com v4/v5 nos mesmos testes:

```
python scripts/evaluate_milk10k.py v6 heldout
python scripts/evaluate_padufes20.py v6
python scripts/evaluate_external.py v6 hiba251
python scripts/evaluate_external.py v6 consec_mel2020_216
python scripts/evaluate_ddi.py v6
python scripts/evaluate_ham_test.py v6
python scripts/threshold_by_modality.py v4 v5 v6
```
